In [1]:
import subprocess
import requests
import json
import pandas as pd
from datetime import datetime, timedelta, timezone
import getpass
import time
from typing import Final

PRECOA_WORKFLOW_ID: Final = "5eafdf4ba6690f01f847b95a"
AUTH_ENDPOINT: Final = "https://appqore.mynglic.com/api/1.0/oauth2/access_token"

def download_files_from_urls(files: list[str], bearer_token: str = None):
    for file in files:
        api_endpoint = f"https://appqore.mynglic.com/api{file}"
    
        filename = file.split("/")[-1]  # Get the last part of the URL after the last '/'

        # Construct the curl command
        command = [
            "curl",
            "-s",  # Suppress progress meter
            "-o", filename,  # Save the downloaded file with the specified filename
            api_endpoint  # The URL to download
        ]
    
        # Add bearer token header if provided
        if bearer_token:
            command.extend(["-H", f"Authorization: Bearer {bearer_token}"])
    
        try:
            subprocess.run(command, check=True)  # check=True raises an exception for non-zero exit codes
    
            print(f"Downloaded '{api_endpoint}' and saved as '{filename}'")
    
        except FileNotFoundError:
            print("Error: 'curl' command not found. Please ensure curl is installed and in your system's PATH.")
        except subprocess.CalledProcessError as e:
            print(f"Error downloading '{url}': {e}")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")


def get_filepath_list(job_details: list[str], bearer_token: str):
    api_endpoint = "https://appqore.mynglic.com/fbu/uapi/bulkOperations/downloadLink"

    files = []
    
    for job_detail in job_details:
    
        command = [
            "curl", "-X", "POST", api_endpoint,
            "-H", f"Authorization: Bearer {bearer_token}",
            "-H", "Content-Type: application/json",
            "-d", json.dumps({"fileLocation": job_detail["destination"]["location"]})
        ]

        try:
            result = subprocess.run(command, capture_output=True, text=True)   
            json_data = json.loads(result.stdout)
            files.append(json_data["downloadLink"])
    
        except FileNotFoundError:
            print("Error: 'curl' command not found. Please ensure curl is installed and in your system's PATH.")
            return None
        except Exception as e:
            print(f"An error occurred: {e}")
            return None
    return files

def get_job_status(download_id: str, bearer_token: str):
    url = f"https://appqore.mynglic.com/fbu/uapi/bulkOperations/jobs/{download_id}"

    command = [
        "curl", 
        "-X", "GET", 
        "-H", f"Authorization: Bearer {bearer_token}",
        "-H", "Content-Type: application/json",
        url
    ]    
    
    try:
        result = subprocess.run(command, capture_output=True, text=True)
        json_data = json.loads(result.stdout)

        return json_data["data"]["status"]
        
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

def get_job_details(download_id: str, bearer_token: str):
    url = f"https://appqore.mynglic.com/fbu/uapi/bulkOperations/jobs/{download_id}"

    command = [
        "curl", 
        "-X", "GET", 
        "-H", f"Authorization: Bearer {bearer_token}",
        "-H", "Content-Type: application/json",
        url
    ]    
    
    try:
        result = subprocess.run(command, capture_output=True, text=True)

        json_data = json.loads(result.stdout)
        steps = json_data["data"]["steps"]
        files = steps[0]["files"]

        return files
        
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

def get_download_id(json_payload: str):
    if output:
        try:
            json_data = json.loads(json_payload)
            if 'id' in json_data:
                return json_data['id']
        except json.JSONDecodeError:
            print(f"Error decoding JSON for hour {hour}. Output: {output}")
            return None

def get_logs_with_curl(bearer_token: str):
    api_endpoint = "https://appqore.mynglic.com/fbu/uapi/bulkOperations/export"

    payload = { 
        "dataModelOrModuleId": PRECOA_WORKFLOW_ID, 
        "dateFieldStartToFilterOn": datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0).strftime("%Y-%m-%dT%H:%M:%S.000Z"), 
        "exportFullRecord": "true", 
        "fileFormat": "JSON" ,
        "name":  "RJWorkflowDataExportTest-Data recorded after this date", 
        "numberOfFilesToGenerate": 1, 
        "targetRecordType": "ALL" 
    }

    json_data = json.dumps(payload)

    command = [
        "curl", "-X", "POST", api_endpoint,
        "-H", f"Authorization: Bearer {bearer_token}",
        "-H", "Content-Type: application/json",
        "-d", json_data
    ]

    try:
        result = subprocess.run(command, capture_output=True, text=True)
        return result.stdout

    except FileNotFoundError:
        print("Error: 'curl' command not found. Please ensure curl is installed and in your system's PATH.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

def get_bearer_token(auth_url, username, password):
    payload = {
        'username': username,
        'password': password,
        'grant_type': 'password' # Common for password grant type in OAuth2
    }
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded' # Often required for this grant type
    }

    try:
        response = requests.post(auth_url, data=payload, headers=headers)
        response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)

        token_data = response.json()
        bearer_token = token_data.get('access_token')

        if bearer_token:
            return bearer_token
        else:
            print(f"Error: 'access_token' not found in response: {token_data}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Error during token request: {e}")
        return None


In [2]:
your_username = input("Please enter your username: ")
your_password = getpass.getpass("Please enter the password: ")
bearer_token = get_bearer_token(AUTH_ENDPOINT, your_username, your_password)

Please enter your username:  jdomingo
Please enter the password:  ········


In [3]:
output = get_logs_with_curl(bearer_token)
jobid = json.loads(output)["id"]

print(f"Job Id: {jobid}")

Job Id: 69dd0e995423c5733e2817b9


In [4]:
file_details_list = []

while True:
    bearer_token = get_bearer_token(AUTH_ENDPOINT, your_username, your_password)
    status = get_job_status(jobid, bearer_token)
    print(f"Current status: {status}")

    # 2. Check the condition to exit
    if status == 'completed':
        # execute_some_code()
        print("Status is complete! retreving the filepaths to download ...")
        file_details_list = get_job_details(jobid, bearer_token)

        print(f"There are {len(file_details_list)} files to download") 
        break
    
    # 3. Wait for 5 minutes (300 seconds) before trying again
    print("Waiting 5 minutes...")
    time.sleep(300)

Current status: created
Waiting 5 minutes...
Current status: inProgress
Waiting 5 minutes...
Current status: inProgress
Waiting 5 minutes...
Current status: inProgress
Waiting 5 minutes...
Current status: inProgress
Waiting 5 minutes...
Current status: completed
Status is complete! retreving the filepaths to download ...
There are 392 files to download


In [5]:
bearer_token = get_bearer_token(AUTH_ENDPOINT, your_username, your_password)

In [6]:
files_to_download = get_filepath_list(file_details_list, bearer_token)

In [ ]:
download_files_from_urls(files_to_download, bearer_token)

In [8]:
if files_to_download:
    files_to_download[:] = [files_to_download[-1]]

In [9]:
files_to_download

['/files/69dd30665423c5733e2818d1']

In [10]:
download_files_from_urls(files_to_download, bearer_token)

Downloaded 'https://appqore.mynglic.com/api/files/69dd30665423c5733e2818d1' and saved as '69dd30665423c5733e2818d1'
